# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [23]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [22]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
#nothing to do with AI , just a function to fetch website links on the given url
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [24]:
# link_system_prompt = """
# You are provided with a list of links found on a webpage.
# You are able to decide which of the links would be most relevant to include in a brochure about the company,
# such as links to an About page, or a Company page, or Careers/Jobs pages.
# You should respond in JSON as in this example:

# {
#     "links": [
#         {"type": "about page", "url": "https://full.url/goes/here/about"},
#         {"type": "careers page", "url": "https://another.full.url/careers"}
#     ]
# }
# """

link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which links on the webpage will be relevant for a person to stay updated on the latest AI technology,
and where it will be heading to and what skills the engineers should focus on to remain ahead, links such as Trends, Whats NExt pages:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

#

In [25]:
#this user prompt is a function because we will be generating a new user prompt based on the different url's that we will be provided
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [26]:
print(get_links_user_prompt("https://thefusepathway.com/blog/the-future-of-ai-5-ai-advancements-to-expect-in-the-next-10-years/?gad_source=1&gad_campaignid=22311785187&gbraid=0AAAAApnnG7wrTqikoBFiA3t8E0RMwFZK_&gclid=Cj0KCQjw37nNBhDkARIsAEBGI8O6yVoEbpj8Qfv2q5Klk4b6B8RmHoUyyslzGps_idpH54hih2-xYS4aAuyuEALw_wcB"))


Here is the list of links on the website https://thefusepathway.com/blog/the-future-of-ai-5-ai-advancements-to-expect-in-the-next-10-years/?gad_source=1&gad_campaignid=22311785187&gbraid=0AAAAApnnG7wrTqikoBFiA3t8E0RMwFZK_&gclid=Cj0KCQjw37nNBhDkARIsAEBGI8O6yVoEbpj8Qfv2q5Klk4b6B8RmHoUyyslzGps_idpH54hih2-xYS4aAuyuEALw_wcB -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#content
https://thefusepathway.com/
https://thefusepathway.com/methodology/
https://thefusepathway.com/book/
https://thefusepathway.com/find-your-passions/
https://thefusepathway.com/studio/robot/
https://thefusepathway.com/studio/robotic-art/
https://vr.thefusepathway.com/?_gl=1*y5zg7v*_gcl_au*NTk2NjQ3NDQyLjE3MzQxMjczODAuMzUzMjkwMjI1LjE3Mzc1NzkyOTEuMTczNzU3OTI5OQ..
https://thefusepathway.com/about/
https://thefusepathway.com/blog/
https://t

In [27]:
#ai call
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"} #tell the api how you want the format to be and inforce technical restriction at inference time
    )
    result = response.choices[0].message.content
    links = json.loads(result) #will turn results into python dictionaries
    return links
    

In [28]:
select_relevant_links("https://thefusepathway.com/blog/the-future-of-ai-5-ai-advancements-to-expect-in-the-next-10-years/?gad_source=1&gad_campaignid=22311785187&gbraid=0AAAAApnnG7wrTqikoBFiA3t8E0RMwFZK_&gclid=Cj0KCQjw37nNBhDkARIsAEBGI8O6yVoEbpj8Qfv2q5Klk4b6B8RmHoUyyslzGps_idpH54hih2-xYS4aAuyuEALw_wcB")

{'links': ['https://thefusepathway.com/',
  'https://thefusepathway.com/about/',
  'https://thefusepathway.com/methodology/',
  'https://thefusepathway.com/book/',
  'https://thefusepathway.com/find-your-passions/',
  'https://thefusepathway.com/studio/',
  'https://thefusepathway.com/studio/robot/',
  'https://thefusepathway.com/studio/robotic-art/',
  'https://thefusepathway.com/about/films/',
  'https://thefusepathway.com/about/media-press/',
  'https://thefusepathway.com/contact/',
  'https://vr.thefusepathway.com/']}

In [29]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [30]:
select_relevant_links("https://thefusepathway.com/blog/the-future-of-ai-5-ai-advancements-to-expect-in-the-next-10-years/?gad_source=1&gad_campaignid=22311785187&gbraid=0AAAAApnnG7wrTqikoBFiA3t8E0RMwFZK_&gclid=Cj0KCQjw37nNBhDkARIsAEBGI8O6yVoEbpj8Qfv2q5Klk4b6B8RmHoUyyslzGps_idpH54hih2-xYS4aAuyuEALw_wcB")

Selecting relevant links for https://thefusepathway.com/blog/the-future-of-ai-5-ai-advancements-to-expect-in-the-next-10-years/?gad_source=1&gad_campaignid=22311785187&gbraid=0AAAAApnnG7wrTqikoBFiA3t8E0RMwFZK_&gclid=Cj0KCQjw37nNBhDkARIsAEBGI8O6yVoEbpj8Qfv2q5Klk4b6B8RmHoUyyslzGps_idpH54hih2-xYS4aAuyuEALw_wcB by calling gpt-5-nano
Found 16 relevant links


{'links': [{'type': 'home page', 'url': 'https://thefusepathway.com/'},
  {'type': 'about page', 'url': 'https://thefusepathway.com/about/'},
  {'type': 'methodology page',
   'url': 'https://thefusepathway.com/methodology/'},
  {'type': 'book page', 'url': 'https://thefusepathway.com/book/'},
  {'type': 'find your passions page',
   'url': 'https://thefusepathway.com/find-your-passions/'},
  {'type': 'studio - robot page',
   'url': 'https://thefusepathway.com/studio/robot/'},
  {'type': 'studio - robotic art page',
   'url': 'https://thefusepathway.com/studio/robotic-art/'},
  {'type': 'VR experiences', 'url': 'https://vr.thefusepathway.com/'},
  {'type': 'about page (overview)',
   'url': 'https://thefusepathway.com/about/'},
  {'type': 'blog page', 'url': 'https://thefusepathway.com/blog/'},
  {'type': 'about - films', 'url': 'https://thefusepathway.com/about/films/'},
  {'type': 'about - media & press',
   'url': 'https://thefusepathway.com/about/media-press/'},
  {'type': 'contac

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [31]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [32]:
print(fetch_page_and_all_relevant_links("https://thefusepathway.com/blog/the-future-of-ai-5-ai-advancements-to-expect-in-the-next-10-years/?gad_source=1&gad_campaignid=22311785187&gbraid=0AAAAApnnG7wrTqikoBFiA3t8E0RMwFZK_&gclid=Cj0KCQjw37nNBhDkARIsAEBGI8O6yVoEbpj8Qfv2q5Klk4b6B8RmHoUyyslzGps_idpH54hih2-xYS4aAuyuEALw_wcB"))

Selecting relevant links for https://thefusepathway.com/blog/the-future-of-ai-5-ai-advancements-to-expect-in-the-next-10-years/?gad_source=1&gad_campaignid=22311785187&gbraid=0AAAAApnnG7wrTqikoBFiA3t8E0RMwFZK_&gclid=Cj0KCQjw37nNBhDkARIsAEBGI8O6yVoEbpj8Qfv2q5Klk4b6B8RmHoUyyslzGps_idpH54hih2-xYS4aAuyuEALw_wcB by calling gpt-5-nano
Found 12 relevant links
## Landing Page:

The Future of AI: 5 AI Advancements to Expect in the Next 10 Years - The Fuse Pathway

Skip to content
What’s Fusioneering?
Book
Quiz
Art & Robotics
Dulcinea
Robot Art
VR Tour
About
Who is Paul Kirby?
Blog
Films
Media & Press
Contact
What’s Fusioneering?
Book
Quiz
Art & Robotics
Dulcinea
Robot Art
VR Tour
About
Who is Paul Kirby?
Blog
Films
Media & Press
Contact
Begin Studio Tour
Assessment
Close Trigger
What’s Fusioneering?
Book
Quiz
Art & Robotics
Dulcinea
Robot Art
VR Tour
About
Who is Paul Kirby?
Blog
Films
Media & Press
Contact
About
Fusioneering
Book
Art
Robot
Learn
About
Fusioneering
Book
Art
Robot
Learn
Begin St

In [33]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

brochure_system_prompt = """
 You are an assistant that analyzes the contents of several relevant pages from a company website
 and creates a short brochure about how to stay updated on the latest AI technology,
and where AI will be heading to and what skills the engineers should focus on to remain ahead
 Include details of latest innovative tech and careers/jobs if you have the information.
 """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("TheFusePathWay", "https://thefusepathway.com/blog/the-future-of-ai-5-ai-advancements-to-expect-in-the-next-10-years/?gad_source=1&gad_campaignid=22311785187&gbraid=0AAAAApnnG7wrTqikoBFiA3t8E0RMwFZK_&gclid=Cj0KCQjw37nNBhDkARIsAEBGI8O6yVoEbpj8Qfv2q5Klk4b6B8RmHoUyyslzGps_idpH54hih2-xYS4aAuyuEALw_wcB")

Selecting relevant links for https://thefusepathway.com/blog/the-future-of-ai-5-ai-advancements-to-expect-in-the-next-10-years/?gad_source=1&gad_campaignid=22311785187&gbraid=0AAAAApnnG7wrTqikoBFiA3t8E0RMwFZK_&gclid=Cj0KCQjw37nNBhDkARIsAEBGI8O6yVoEbpj8Qfv2q5Klk4b6B8RmHoUyyslzGps_idpH54hih2-xYS4aAuyuEALw_wcB by calling gpt-5-nano
Found 20 relevant links


'\nYou are looking at a company called: The Fuse PathWay\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nThe Future of AI: 5 AI Advancements to Expect in the Next 10 Years - The Fuse Pathway\n\nSkip to content\nWhat’s Fusioneering?\nBook\nQuiz\nArt & Robotics\nDulcinea\nRobot Art\nVR Tour\nAbout\nWho is Paul Kirby?\nBlog\nFilms\nMedia & Press\nContact\nWhat’s Fusioneering?\nBook\nQuiz\nArt & Robotics\nDulcinea\nRobot Art\nVR Tour\nAbout\nWho is Paul Kirby?\nBlog\nFilms\nMedia & Press\nContact\nBegin Studio Tour\nAssessment\nClose Trigger\nWhat’s Fusioneering?\nBook\nQuiz\nArt & Robotics\nDulcinea\nRobot Art\nVR Tour\nAbout\nWho is Paul Kirby?\nBlog\nFilms\nMedia & Press\nContact\nAbout\nFusioneering\nBook\nArt\nRobot\nLearn\nAbout\nFusioneering\nBook\nArt\nRobot\nLearn\nBegin Studio Tour\nAssessment\nClose Trigger\nAbout\nFusioneering\nVR Tour\nB

In [36]:
#second llm call, don't need to put in a 3rd parameter in chat completions as we want the response in plain english
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [41]:
create_brochure("TheFusePathway", "https://thefusepathway.com/blog/the-future-of-ai-5-ai-advancements-to-expect-in-the-next-10-years/?gad_source=1&gad_campaignid=22311785187&gbraid=0AAAAApnnG7wrTqikoBFiA3t8E0RMwFZK_&gclid=Cj0KCQjw37nNBhDkARIsAEBGI8O6yVoEbpj8Qfv2q5Klk4b6B8RmHoUyyslzGps_idpH54hih2-xYS4aAuyuEALw_wcB")

Selecting relevant links for https://thefusepathway.com/blog/the-future-of-ai-5-ai-advancements-to-expect-in-the-next-10-years/?gad_source=1&gad_campaignid=22311785187&gbraid=0AAAAApnnG7wrTqikoBFiA3t8E0RMwFZK_&gclid=Cj0KCQjw37nNBhDkARIsAEBGI8O6yVoEbpj8Qfv2q5Klk4b6B8RmHoUyyslzGps_idpH54hih2-xYS4aAuyuEALw_wcB by calling gpt-5-nano
Found 16 relevant links


TypeError: string indices must be integers, not 'str'

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [42]:
#can pass in stream = true, to visibly see the llm write the response like typewrting 
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [43]:
stream_brochure("TheFuseWay", "https://thefusepathway.com/blog/the-future-of-ai-5-ai-advancements-to-expect-in-the-next-10-years/?gad_source=1&gad_campaignid=22311785187&gbraid=0AAAAApnnG7wrTqikoBFiA3t8E0RMwFZK_&gclid=Cj0KCQjw37nNBhDkARIsAEBGI8O6yVoEbpj8Qfv2q5Klk4b6B8RmHoUyyslzGps_idpH54hih2-xYS4aAuyuEALw_wcB")

Selecting relevant links for https://thefusepathway.com/blog/the-future-of-ai-5-ai-advancements-to-expect-in-the-next-10-years/?gad_source=1&gad_campaignid=22311785187&gbraid=0AAAAApnnG7wrTqikoBFiA3t8E0RMwFZK_&gclid=Cj0KCQjw37nNBhDkARIsAEBGI8O6yVoEbpj8Qfv2q5Klk4b6B8RmHoUyyslzGps_idpH54hih2-xYS4aAuyuEALw_wcB by calling gpt-5-nano
Found 16 relevant links


# TheFuseWay: Your Pathway to the Future of AI

## Who We Are
TheFuseWay is a pioneering company dedicated to exploring and shaping the future of artificial intelligence. With a passion for innovation, creativity, and education, TheFuseWay offers a unique blend of technology, art, and robotics designed to inspire and equip individuals and engineers for the AI-driven world of tomorrow.

## Our Vision: The Future of AI in the Next Decade
At TheFuseWay, we focus on the "Fuse Pathway"—a journey through the most exciting advancements in AI technology expected over the next 10 years. We believe that understanding these trends is crucial for anyone wanting to stay ahead in the rapidly evolving tech landscape.

### Five AI Advancements to Watch:
- The integration of AI with robotics and art, blending creativity and technology.
- Advances in virtual and augmented reality powered by AI, creating immersive experiences.
- Development of new AI-driven tools for learning and self-assessment.
- Innovations in robot-assisted creativity and autonomous art creation.
- Expansion of AI applications across diverse industries, reshaping career paths and skill requirements.

## Stay Updated with TheFuseWay
- **Book:** Discover in-depth insights in our featured book capturing the essence of AI's future.
- **Quiz & Assessment:** Engage with interactive tools to evaluate your skills and readiness for the AI era.
- **Blog & Films:** Access expert articles and documentaries focused on AI developments and stories.
- **VR Tour:** Experience immersive virtual reality environments combined with AI storytelling.
- **Media & Press:** Follow our latest news, interviews, and media coverage to stay informed.

## Skills and Careers for the Future
Engineers and AI professionals should focus on the following to remain at the forefront:
- **Fusioneering:** Mastering the fusion of AI with other disciplines such as art and robotics.
- **Creative AI Development:** Designing AI systems that enhance creativity and can collaborate with humans.
- **AI & Robotics Engineering:** Building intelligent autonomous systems for practical and artistic applications.
- **Virtual Reality & AI Integration:** Creating immersive, adaptive environments powered by AI.
- **Continuous Learning & Adaptability:** Using AI-driven assessment tools to upskill and stay relevant.

## Why Choose TheFuseWay?
- A holistic approach combining cutting-edge AI technology with artistic expression.
- Unique educational resources tailored to future AI advancements.
- A community and network of innovators, engineers, and creatives passionate about AI.
- Complimentary experiences like VR tours and interactive assessments to deepen understanding.

---

**Connect with TheFuseWay and Prepare for Tomorrow Today**  
Explore more, learn, and innovate at [TheFuseWay Website](#)  
Contact us to start your journey into the future of AI.

---

*TheFuseWay: Bridging Creativity and Technology for the Next Generation of AI Innovators.*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>